In [1]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# --- Настройки ---------------------------------------------------------------
AVRO_PATH = "activity_log.avro"

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

SPARK_AVRO_PACKAGE = "org.apache.spark:spark-avro_2.12:3.5.5"

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("task3_user_sessions")
    .config("spark.jars.packages", SPARK_AVRO_PACKAGE)
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

raw = spark.read.format("avro").load(AVRO_PATH)

df = raw.withColumn("event_time", F.to_timestamp("timestamp", "dd-MMM-yyyy HH:mm"))

print("Схема:")
df.printSchema()
df.show(truncate=False)


Схема:
root
 |-- event_id: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- page_url: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- event_time: timestamp (nullable = true)

+--------+-------+----------+------------+-----------------+--------------------+----------+------+-------------------+
|event_id|user_id|session_id|event_type  |timestamp        |page_url            |product_id|amount|event_time         |
+--------+-------+----------+------------+-----------------+--------------------+----------+------+-------------------+
|E001    |101    |S101-A    |login       |01-Jan-2025 10:00|/login              |NULL      |NULL  |2025-01-01 10:00:00|
|E002    |101    |S101-A    |view_page   |01-Jan-2025 10:05|/products/category_A|NULL      |NULL  |2025-01-01 10:05:00|
|E003    |101    

In [2]:
# --- Задание 1: ежедневная активность -----------------------------------------
print("Задание 1")
daily = (
    df.groupBy(F.to_date("event_time").alias("date"))
    .agg(F.count("event_id").alias("total_events"))
    .orderBy("date")
)
daily.show()


Задание 1
+----------+------------+
|      date|total_events|
+----------+------------+
|2025-01-01|           6|
|2025-01-02|           3|
|2025-01-03|           3|
|2025-01-04|           2|
|2025-01-05|           6|
|2025-01-06|           5|
|2025-01-07|           5|
|2025-01-08|           4|
|2025-01-09|           6|
+----------+------------+



In [3]:
# --- Задание 2: пользователи и сессии -----------------------------------------
print("Задание 2")
print("Количество уникальных пользователей")
df.select("user_id").distinct().orderBy("user_id").show()

# Сессия уникальна в паре (user_id, session_id)
total_sessions = df.select("user_id", "session_id").distinct().count()
print("Количество уникальных сессий")
print(total_sessions)

(
    df.groupBy("user_id")
    .agg(F.countDistinct("session_id").alias("unique_sessions_count"))
    .orderBy("user_id")
    .show()
)


Задание 2
Количество уникальных пользователей
+-------+
|user_id|
+-------+
|    101|
|    102|
|    103|
|    104|
|    105|
|    106|
|    107|
+-------+

Количество уникальных сессий
9
+-------+---------------------+
|user_id|unique_sessions_count|
+-------+---------------------+
|    101|                    2|
|    102|                    2|
|    103|                    1|
|    104|                    1|
|    105|                    1|
|    106|                    1|
|    107|                    1|
+-------+---------------------+



In [4]:
# --- Задание 3: детализация сессий --------------------------------------------
print("Задание 3")
(
    df.groupBy("user_id", "session_id")
    .agg(F.count("event_id").alias("count_events_in_session"))
    # user_id, session_id добавлены, чтобы порядок при равных count был стабильным
    .orderBy("count_events_in_session", "user_id", "session_id")
    .show()
)


Задание 3
+-------+----------+-----------------------+
|user_id|session_id|count_events_in_session|
+-------+----------+-----------------------+
|    104|    S104-D|                      2|
|    102|    S102-B|                      3|
|    103|    S103-C|                      3|
|    106|    S106-F|                      4|
|    101|    S101-B|                      5|
|    102|    S102-C|                      5|
|    101|    S101-A|                      6|
|    105|    S105-E|                      6|
|    107|    S107-G|                      6|
+-------+----------+-----------------------+



In [5]:
# --- Задание 4: покупки -------------------------------------------------------
print("Задание 4")
purchases = df.filter(F.col("event_type") == "purchase")

stats = purchases.agg(
    F.count("*").alias("count_purchase_amount"),
    F.sum("amount").alias("sum_purchase_amount"),
    F.round(F.avg("amount"), 2).alias("average_purchase_amount"),
)
stats.select("count_purchase_amount").show()
stats.select("sum_purchase_amount").show()
stats.select("average_purchase_amount").show()


Задание 4
+---------------------+
|count_purchase_amount|
+---------------------+
|                    4|
+---------------------+

+-------------------+
|sum_purchase_amount|
+-------------------+
|              905.5|
+-------------------+

+-----------------------+
|average_purchase_amount|
+-----------------------+
|                 226.38|
+-----------------------+



In [6]:
# --- Задание 5: продолжительность сессий --------------------------------------
print("Задание 5")
sessions = (
    df.groupBy("user_id", "session_id")
    .agg(
        F.min("event_time").alias("min_time"),
        F.max("event_time").alias("max_time"),
    )
    .withColumn(
        "session_duration_seconds",
        F.unix_timestamp("max_time") - F.unix_timestamp("min_time"),
    )
    .orderBy("user_id", "session_id")
)
sessions.show(truncate=False)

sessions.agg(
    F.round(F.avg("session_duration_seconds"), 2).alias("average_session_duration_seconds")
).show()

spark.stop()

Задание 5
+-------+----------+-------------------+-------------------+------------------------+
|user_id|session_id|min_time           |max_time           |session_duration_seconds|
+-------+----------+-------------------+-------------------+------------------------+
|101    |S101-A    |2025-01-01 10:00:00|2025-01-01 10:20:00|1200                    |
|101    |S101-B    |2025-01-06 09:30:00|2025-01-06 09:40:00|600                     |
|102    |S102-B    |2025-01-02 11:30:00|2025-01-02 11:38:00|480                     |
|102    |S102-C    |2025-01-07 15:00:00|2025-01-07 15:18:00|1080                    |
|103    |S103-C    |2025-01-03 09:00:00|2025-01-03 09:10:00|600                     |
|104    |S104-D    |2025-01-04 14:00:00|2025-01-04 14:03:00|180                     |
|105    |S105-E    |2025-01-05 16:45:00|2025-01-05 17:02:00|1020                    |
|106    |S106-F    |2025-01-08 10:00:00|2025-01-08 10:10:00|600                     |
|107    |S107-G    |2025-01-09 11:00:00|2025